# Workshop 6 — Multi-Step Agent: Reason, Act, Observe with NVIDIA NIM

Workshop 5 taught the model to pick **one** tool and answer. That's a chatbot with hands. A real agent has to **chain** tools — think, call a tool, read the result, decide what to do next, and only then answer. That loop is the **ReAct** pattern (Reason + Act), and this notebook makes it visible so you can watch the agent work through a question one step at a time.

We add a third tool — a date calculator — specifically to force multi-step reasoning. A question like *"how many days until the next AI Club meeting?"* can't be answered by any single tool: the agent has to **search** for the meeting day, **then** compute the days until it.

Same hosted NIM endpoint and same `nvidia/llama-3.3-nemotron-super-49b-v1.5` model as Workshop 5 — this NVIDIA-tuned model is far more reliable once the agent has to choose *and sequence* multiple tools.

## Step 1 — Setup (self-contained)

Bundles the retriever from Workshop 2 and the tool-calling setup from Workshop 5 so this notebook stands on its own. Paste your NVIDIA API key when prompted.

In [ ]:
%pip install -q openai numpy

import os, getpass, json
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from openai import OpenAI
import numpy as np

if not os.getenv('NVIDIA_API_KEY'):
    os.environ['NVIDIA_API_KEY'] = getpass.getpass('Paste your NVIDIA API key (starts with nvapi-): ')

client = OpenAI(
    base_url='https://integrate.api.nvidia.com/v1',
    api_key=os.environ['NVIDIA_API_KEY'],
)

MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1.5'   # same as Workshop 5 — reliable multi-tool calling
EMBED_MODEL = 'nvidia/nv-embedqa-e5-v5'
LOCAL_TZ = 'America/Los_Angeles'        # USC campus time zone

knowledge_base = [
    {'title': 'USC AI Club meeting',
     'text': 'The USC AI Club meets every Thursday at 5 PM in the engineering building, room 204.'},
    {'title': 'USC GPU lab hours',
     'text': 'The USC GPU computing lab is open Monday to Friday from 10 AM to 6 PM.'},
    {'title': 'NVIDIA Developer Program',
     'text': 'USC students can join the NVIDIA Developer Program for free.'},
    {'title': 'Next USC workshop',
     'text': 'The next USC AI Club workshop will cover Retrieval Augmented Generation (RAG).'},
    {'title': 'USC AI/ML office hours',
     'text': 'Office hours for the USC AI/ML faculty are Tuesdays 2-4 PM.'},
    {'title': 'USC robotics lab',
     'text': 'The USC robotics lab requires safety training before students can use the soldering station.'},
    {'title': 'USC tutoring',
     'text': 'Peer tutoring for introductory Python at USC is available Wednesdays from 1 PM to 3 PM.'},
]

def embed_texts(texts, input_type='passage'):
    response = client.embeddings.create(model=EMBED_MODEL, input=texts, extra_body={'input_type': input_type})
    return [np.array(item.embedding, dtype=np.float32) for item in response.data]

def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

def retrieve_context(question, k=3):
    q_emb = embed_texts([question], input_type='query')[0]
    scored = [(cosine_similarity(q_emb, item['embedding']), item) for item in knowledge_base]
    scored.sort(key=lambda p: p[0], reverse=True)
    return '\n'.join(f"- {item['text']}" for _, item in scored[:k])

for item, emb in zip(knowledge_base, embed_texts([i['text'] for i in knowledge_base], 'passage')):
    item['embedding'] = emb
print(f'Ready. Embedded {len(knowledge_base)} chunks. Model: {MODEL}')

## Step 2 — Three tools (one of them forces chaining)

The retriever and the clock you already know from Workshop 5. The new one is `days_until_weekday`: it can't answer a question on its own, because the agent first has to *find out which day* an event happens. That dependency is exactly what turns a one-shot tool call into a multi-step plan.

In [ ]:
WEEKDAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

def get_current_time(timezone=LOCAL_TZ):
    try:
        zone = ZoneInfo(timezone)
    except Exception:
        zone = ZoneInfo('UTC')
    return datetime.now(zone).strftime('%A, %B %d, %Y at %I:%M %p %Z')

def search_campus_info(query):
    return retrieve_context(query, k=3)   # reuse the Workshop 2 retriever

def days_until_weekday(weekday):
    target = weekday.strip().capitalize()
    if target not in WEEKDAYS:
        return f"'{weekday}' is not a valid weekday. Use one of: {', '.join(WEEKDAYS)}."
    today = datetime.now(ZoneInfo(LOCAL_TZ))
    delta = (WEEKDAYS.index(target) - today.weekday()) % 7
    date_str = (today + timedelta(days=delta)).strftime('%B %d, %Y')
    if delta == 0:
        return f"Today is {target} ({date_str}) — that is 0 days away (it's today)."
    return f"The next {target} is in {delta} day(s), on {date_str}. Today is {today.strftime('%A')}."

# Quick smoke test
print(get_current_time())
print(days_until_weekday('Thursday'))
print(search_campus_info('AI Club'))

## Step 3 — Describe the tools to the model

The schema is what the model reads to decide which tool to call and when. Notice the `days_until_weekday` description tells the model it *usually has to search first* — guiding the model toward the right sequence is part of the prompt engineering.

In [ ]:
tools = [
    {'type': 'function', 'function': {
        'name': 'search_campus_info',
        'description': 'Search the USC campus knowledge base for facts about USC clubs (including the AI Club), labs (GPU lab, robotics lab), workshops, faculty office hours, peer tutoring, and the NVIDIA Developer Program. Use this to find WHEN or WHERE something happens. Always call this for any USC-specific fact — do not answer from your own knowledge.',
        'parameters': {
            'type': 'object',
            'properties': {'query': {'type': 'string', 'description': 'The USC campus question or search phrase.'}},
            'required': ['query'],
        },
    }},
    {'type': 'function', 'function': {
        'name': 'get_current_time',
        'description': "Get the current date, day of week, and time in an IANA time zone. Use this when the question depends on what day or time it is right now (e.g. 'is the lab open right now?').",
        'parameters': {
            'type': 'object',
            'properties': {'timezone': {'type': 'string', 'description': 'IANA time zone, e.g. America/Los_Angeles or UTC.'}},
        },
    }},
    {'type': 'function', 'function': {
        'name': 'days_until_weekday',
        'description': 'Calculate how many days from today until the next occurrence of a given weekday (e.g. Thursday). Use this AFTER you know which day an event happens, to work out how far away it is. You usually have to call search_campus_info first to find the day.',
        'parameters': {
            'type': 'object',
            'properties': {'weekday': {'type': 'string', 'description': 'A weekday name, e.g. Monday, Thursday, Sunday.'}},
            'required': ['weekday'],
        },
    }},
]

available_tools = {
    'search_campus_info': search_campus_info,
    'get_current_time': get_current_time,
    'days_until_weekday': days_until_weekday,
}

## Step 4 — The ReAct loop

Same skeleton as Workshop 5, with two changes that make it a real agent:

1. **A bigger step budget** (`MAX_STEPS = 5`) — multi-step questions need room to call several tools in a row.
2. **A visible trace** — each iteration prints the tool it called (*act*) and what came back (*observe*), so you can read the agent's reasoning as control flow.

The system prompt explicitly tells the model to work in a loop and to chain tools (search for the day, *then* compute days until it).

In [ ]:
SYSTEM_PROMPT = (
    '/no_think\n\nYou are a USC campus assistant that solves questions step by step using tools. '
    'You have three tools: search_campus_info (find USC facts), get_current_time '
    "(today's date and time), and days_until_weekday (days from today until a weekday).\n\n"
    'Work in a loop: think about what you still need, call ONE tool to get it, read the '
    'result, then decide whether you can answer or need another tool. Many questions need '
    'more than one tool — for example, to find how many days until an event, first search '
    'for the day it happens, then call days_until_weekday with that day.\n\n'
    'Only answer once you have gathered every fact you need from the tools. Base your final '
    'answer strictly on tool results, never on your own assumptions about USC. If the tools '
    'cannot give you the answer, reply exactly: '
    "I don't have that information — check with the USC AI Club."
)

MAX_STEPS = 5   # multi-step questions need more room than Workshop 5's cap of 3

def run_agent(question, verbose=True):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    for step in range(1, MAX_STEPS + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice='auto', temperature=0.2, max_tokens=400,
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:                 # model is done → final answer
            return message.content

        for tool_call in message.tool_calls:        # run every tool it asked for
            name = tool_call.function.name
            try:
                arguments = json.loads(tool_call.function.arguments or '{}')
            except json.JSONDecodeError:
                arguments = {}
            if name not in available_tools:
                result = f"Tool '{name}' is not available."
            else:
                try:
                    result = available_tools[name](**arguments)
                except Exception as exc:            # a bad tool call must not crash the loop
                    result = f"Tool '{name}' failed: {exc}"
            if verbose:
                print(f'  step {step} · acting  -> {name}({json.dumps(arguments)})')
                print(f'  step {step} · observe <- {result}')
            messages.append({'role': 'tool', 'tool_call_id': tool_call.id, 'name': name, 'content': str(result)})

    return 'I reached the step limit before finishing — try asking a narrower question.'

## Step 5 — Run it and watch the steps

In [ ]:
for question in [
    'How many days until the next USC AI Club meeting?',   # search (Thursday) -> days_until_weekday
    'Is the USC GPU lab open right now?',                  # get_current_time + search, then reason
    'When does the USC AI Club meet?',                     # one tool is enough — should NOT over-call
    'What is the campus wifi password?',                   # no tool can answer — should refuse
]:
    print(f'Q: {question}')
    print(f'A: {run_agent(question, verbose=True)}\n')

## What just happened

Read the trace, not just the final answers:

- **"How many days until the next AI Club meeting?"** — the agent calls `search_campus_info`, reads *"every Thursday"*, then calls `days_until_weekday('Thursday')`, and only then answers. Two tools, in order, with the second depending on the first. That's the whole point of Workshop 6.
- **"Is the GPU lab open right now?"** — it checks `get_current_time` for today's day/hour and `search_campus_info` for the posted hours (Mon–Fri, 10 AM–6 PM), then reasons about whether *now* falls inside that window.
- **"When does the AI Club meet?"** — one search is enough. A good agent doesn't call tools it doesn't need.
- **"What is the wifi password?"** — it searches, finds nothing, and falls back to the refusal line. The Workshop 3 guardrail still holds, now inside a multi-step loop.

## The full series

- **Workshop 1:** First API call — the chat function.
- **Workshop 2:** Embedding-based RAG — the retriever this agent reuses.
- **Workshop 3:** Guardrails — the refusal pattern that still holds here.
- **Workshop 4:** Run NIM on your own GPU — same code, different endpoint.
- **Workshop 5:** Tool calling — chatbot picks one tool.
- **Workshop 6 (this notebook):** Multi-step ReAct — the agent chains tools and reasons across steps.

The spine never changed: an LLM is a normal Python function, and everything around it — retrieval, guardrails, deployment, tool calling, the ReAct loop — is normal software you control.

Star and fork the repo for your school: https://github.com/torkian/nvidia-nim-workshop